# 🌱 Kaggle S6E4 — Predicting Irrigation Need
**Metric:** Balanced Accuracy | **Classes:** Low / Medium / High | **Deadline:** 30. April 2026

Pipeline: EDA → Feature Engineering → LightGBM + CatBoost (5-Fold CV) → Weighted Ensemble → Submission

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

SEED = 42
# ── Run-Profile ──
# Schnell (default, ~1.5–2h on Kaggle):  SEEDS=[42],           N_SPLITS=5, ITERS=1200
# Mittel (~3–4h on Kaggle):              SEEDS=[42, 2024],     N_SPLITS=5, ITERS=1500
# Final (~6–9h on Kaggle):               SEEDS=[42, 2024, 7],  N_SPLITS=5, ITERS=3000
SEEDS    = [42]
N_SPLITS = 5
N_ITERS  = 1200
TARGET   = 'Irrigation_Need'

# Pseudo-Labeling verdoppelt die Trainingszeit. Erst aktivieren wenn
# Round-1-Ensemble deutlich besser als Single-Modelle → Signal stark genug.
USE_PSEUDO_LABELING   = False
PSEUDO_CONF_THRESHOLD = 0.97

print('Libraries loaded ✅')
print(f'Profil: {len(SEEDS)} seeds × {N_SPLITS} folds × {N_ITERS} iters | pseudo-labels: {USE_PSEUDO_LABELING}')

## 1. Load Data

In [ ]:
import os, glob

# ── Auto-detect Kaggle vs. local environment ──
KAGGLE_PATH = '/kaggle/input/competitions/playground-series-s6e4'
if os.path.exists(KAGGLE_PATH):
    DATA_DIR = KAGGLE_PATH
else:
    DATA_DIR = '.'
print(f'DATA_DIR: {DATA_DIR}')

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

# ── Find external miadul dataset anywhere under /kaggle/input ──
# Kaggle mounts additional datasets at /kaggle/input/<slug>/ — the exact slug
# depends on how the user added it. We search recursively for the CSV.
external_loaded = False
search_paths = [
    '/kaggle/input/**/irrigation_prediction.csv',
    '/kaggle/input/**/irrigation*prediction*.csv',
    '/kaggle/input/**/*irrigation_water*.csv',
    './irrigation_prediction.csv',
    './external/irrigation_prediction.csv',
]
for pattern in search_paths:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        external_path = matches[0]
        external_loaded = True
        print(f'✅ External dataset found: {external_path}')
        break

if external_loaded:
    original = pd.read_csv(external_path)
    # Flag samples by origin
    train['is_generated']    = 1
    test['is_generated']     = 1
    original['is_generated'] = 0
    # Combine: only columns present in both
    common_cols = [c for c in train.columns if c in original.columns]
    print(f'   rows: {original.shape[0]}, matching cols: {len(common_cols)}')
    train = pd.concat([train, original[common_cols]], ignore_index=True)
    initial_len = len(train)
    train = train.drop_duplicates().reset_index(drop=True)
    dropped = initial_len - len(train)
    if dropped:
        print(f'   dropped {dropped} duplicate rows')
else:
    print('⚠️  External miadul dataset NOT found — running on synthetic data only.')
    print('   To add it: Kaggle Notebook → Add Input → search "miadul irrigation"')
    # Do NOT add the is_generated column — it would be a constant feature.

print(f'\nTrain: {train.shape}')
print(f'Test:  {test.shape}')
print(f'Sub:   {sub.shape}')
if 'is_generated' in train.columns:
    print(f"is_generated dist: {train['is_generated'].value_counts().to_dict()}")
train.head()

## 2. EDA

In [ ]:
print('=== Data Types ===')
print(train.dtypes)
print('\n=== Missing Values ===')
print(train.isnull().sum())
print('\n=== Target Distribution ===')
print(train[TARGET].value_counts())
print(train[TARGET].value_counts(normalize=True).round(3))

In [ ]:
print('=== Numeric Summary ===')
train.describe().T

In [ ]:
# Target class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = train[TARGET].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2ecc71','#f39c12','#e74c3c'])
axes[0].set_title('Target Distribution (absolute)', fontsize=13)
axes[0].set_xlabel('Irrigation Need')
axes[0].set_ylabel('Count')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 100, f'{val:,}', ha='center', fontsize=10)

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2ecc71','#f39c12','#e74c3c'], startangle=90)
axes[1].set_title('Target Distribution (%)', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric features)
num_cols = train.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != 'id']

if len(num_cols) > 1:
    plt.figure(figsize=(14, 10))
    corr = train[num_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap='RdYlGn', center=0,
                annot=len(num_cols) <= 15, fmt='.2f', linewidths=0.5)
    plt.title('Feature Correlation Matrix', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution of numeric features by target
num_features = [c for c in num_cols if c not in ['id']]

n_cols = 3
n_rows = (len(num_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()

colors = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}

for i, col in enumerate(num_features):
    ax = axes[i]
    for label in train[TARGET].unique():
        subset = train[train[TARGET] == label][col].dropna()
        subset.hist(ax=ax, bins=40, alpha=0.6, label=label,
                    color=colors.get(label, 'gray'), density=True)
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by Irrigation Need Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features overview
cat_cols = train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != TARGET]
print(f'Categorical columns: {cat_cols}')
for col in cat_cols:
    print(f'\n{col}: {train[col].nunique()} unique values')
    print(train[col].value_counts().head(10))

## 3. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    cols = df.columns.tolist()

    # ── Interaction Features (echte Spaltennamen aus Kaggle S6E4) ──
    interaction_pairs = [
        ('Temperature_C',          'Humidity'),
        ('Temperature_C',          'Soil_Moisture'),
        ('Humidity',               'Soil_Moisture'),
        ('Wind_Speed_kmh',         'Temperature_C'),
        ('Sunlight_Hours',         'Temperature_C'),
        ('Sunlight_Hours',         'Humidity'),
        ('Rainfall_mm',            'Soil_Moisture'),
        ('Rainfall_mm',            'Humidity'),
        ('Soil_pH',                'Organic_Carbon'),
        ('Soil_pH',                'Electrical_Conductivity'),
        ('Organic_Carbon',         'Electrical_Conductivity'),
        ('Previous_Irrigation_mm', 'Soil_Moisture'),
        ('Previous_Irrigation_mm', 'Rainfall_mm'),
        ('Field_Area_hectare',     'Previous_Irrigation_mm'),
    ]
    for a, b in interaction_pairs:
        if a in cols and b in cols:
            df[f'{a}_x_{b}']     = df[a] * df[b]
            df[f'{a}_minus_{b}'] = df[a] - df[b]
            df[f'{a}_div_{b}']   = df[a] / (df[b].abs() + 1e-9)

    # ── Domain-spezifische Indizes ──
    if {'Rainfall_mm', 'Previous_Irrigation_mm', 'Soil_Moisture'}.issubset(cols):
        df['water_balance'] = (df['Rainfall_mm'] + df['Previous_Irrigation_mm']) - df['Soil_Moisture']
    if {'Temperature_C', 'Humidity'}.issubset(cols):
        # einfache "trockenheits"-proxy: hohe Temp + niedrige Humidity
        df['dryness_index'] = df['Temperature_C'] * (100 - df['Humidity']) / 100
    if {'Sunlight_Hours', 'Rainfall_mm'}.issubset(cols):
        df['sun_rain_ratio'] = df['Sunlight_Hours'] / (df['Rainfall_mm'].abs() + 1e-9)

    # ── Polynomial Features für numerische Cols ──
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    num_cols = [c for c in num_cols if c not in ['id'] and '_x_' not in c
                and '_minus_' not in c and '_div_' not in c]
    for col in num_cols:
        df[f'{col}_sq']    = df[col] ** 2
        df[f'{col}_log1p'] = np.log1p(df[col].clip(lower=0))

    # ── Aggregate Stats ──
    base_numeric = [c for c in num_cols if c != 'id']
    if len(base_numeric) > 2:
        df['row_mean']  = df[base_numeric].mean(axis=1)
        df['row_std']   = df[base_numeric].std(axis=1)
        df['row_max']   = df[base_numeric].max(axis=1)
        df['row_min']   = df[base_numeric].min(axis=1)
        df['row_range'] = df['row_max'] - df['row_min']

    return df

train = engineer_features(train)
test  = engineer_features(test)

print(f'Train nach FE: {train.shape}')
print(f'Test  nach FE: {test.shape}')
print(f'Neue Features: {train.shape[1] - 21}')  # 21 = original cols inkl. target

## 4. Encoding

In [ ]:
# Target encoding
le_target = LabelEncoder()
train[TARGET] = le_target.fit_transform(train[TARGET])
print(f'Class mapping: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}')

# Categorical features encoding
cat_cols = train.select_dtypes(include='object').columns.tolist()
print(f'Encoding categoricals: {cat_cols}')

for col in cat_cols:
    le_col = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le_col.fit(combined)
    train[col] = le_col.transform(train[col].astype(str))
    test[col]  = le_col.transform(test[col].astype(str))

FEATURES = [c for c in train.columns if c not in ['id', TARGET]]
X      = train[FEATURES]
y      = train[TARGET]
X_test = test[FEATURES]

print(f'Features: {len(FEATURES)}')
print(f'X: {X.shape} | y: {y.shape} | X_test: {X_test.shape}')

## 5. LightGBM — 5-Fold Stratified CV

In [ ]:
# LightGBM training as a function so we can re-run after pseudo-labeling
lgb_params_base = {
    'objective':         'multiclass',
    'num_class':         3,
    'metric':            'multi_logloss',
    'learning_rate':     0.03,            # niedriger als vorher (0.05)
    'num_leaves':        63,              # weniger als 127 → weniger overfit
    'max_depth':         7,
    'min_child_samples': 50,              # mehr Regularisierung
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      1,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'n_estimators':      N_ITERS,
    'verbose':           -1,
    'class_weight':      'balanced',
}

def train_lgb(X_in, y_in, X_test_in, sample_weight=None):
    oof   = np.zeros((len(X_in), 3))
    preds = np.zeros((len(X_test_in), 3))
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        fold_scores = []
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_in, y_in)):
            X_tr, X_val = X_in.iloc[tr_idx], X_in.iloc[val_idx]
            y_tr, y_val = y_in.iloc[tr_idx], y_in.iloc[val_idx]
            sw_tr = sample_weight[tr_idx] if sample_weight is not None else None

            model = lgb.LGBMClassifier(**lgb_params_base, random_state=seed)
            model.fit(
                X_tr, y_tr, sample_weight=sw_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
            )
            oof[val_idx] += model.predict_proba(X_val) / len(SEEDS)
            preds        += model.predict_proba(X_test_in) / (N_SPLITS * len(SEEDS))
            fold_scores.append(balanced_accuracy_score(
                y_val, np.argmax(model.predict_proba(X_val), axis=1)))
        print(f'  LGB seed {seed} | folds: {[round(s,5) for s in fold_scores]}')
    return oof, preds, model  # last model for feat-importance

print('── LightGBM Round 1 ──')
lgb_oof, lgb_preds, lgb_last_model = train_lgb(X, y, X_test)
lgb_cv = balanced_accuracy_score(y, np.argmax(lgb_oof, axis=1))
print(f'\n✅ LightGBM OOF: {lgb_cv:.5f}')

In [ ]:
# Feature importance from last LGB model
importances = pd.DataFrame({
    'feature': FEATURES,
    'importance': lgb_last_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
top_n = min(30, len(importances))
sns.barplot(data=importances.head(top_n), x='importance', y='feature', palette='viridis')
plt.title(f'LightGBM — Top {top_n} Feature Importances', fontsize=13)
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(importances.head(10).to_string(index=False))

## 6. CatBoost — 5-Fold Stratified CV

In [ ]:
cb_params_base = dict(
    iterations=N_ITERS,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=5,
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Macro',
    early_stopping_rounds=100,
    verbose=0,
    auto_class_weights='Balanced',
    task_type='CPU',
)

def train_cb(X_in, y_in, X_test_in):
    oof   = np.zeros((len(X_in), 3))
    preds = np.zeros((len(X_test_in), 3))
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        fold_scores = []
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_in, y_in)):
            X_tr, X_val = X_in.iloc[tr_idx], X_in.iloc[val_idx]
            y_tr, y_val = y_in.iloc[tr_idx], y_in.iloc[val_idx]

            model = CatBoostClassifier(**cb_params_base, random_seed=seed)
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
            oof[val_idx] += model.predict_proba(X_val) / len(SEEDS)
            preds        += model.predict_proba(X_test_in) / (N_SPLITS * len(SEEDS))
            fold_scores.append(balanced_accuracy_score(
                y_val, np.argmax(model.predict_proba(X_val), axis=1)))
        print(f'  CB seed {seed} | folds: {[round(s,5) for s in fold_scores]}')
    return oof, preds

print('── CatBoost Round 1 ──')
cb_oof, cb_preds = train_cb(X, y, X_test)
cb_cv = balanced_accuracy_score(y, np.argmax(cb_oof, axis=1))
print(f'\n✅ CatBoost OOF: {cb_cv:.5f}')

## 6b. XGBoost — 5-Fold Stratified CV

In [ ]:
xgb_params_base = dict(
    objective='multi:softprob',
    num_class=3,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    n_estimators=N_ITERS,
    eval_metric='mlogloss',
    early_stopping_rounds=100,
    tree_method='hist',
    verbosity=0,
)

def train_xgb(X_in, y_in, X_test_in):
    oof   = np.zeros((len(X_in), 3))
    preds = np.zeros((len(X_test_in), 3))
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        fold_scores = []
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_in, y_in)):
            X_tr, X_val = X_in.iloc[tr_idx], X_in.iloc[val_idx]
            y_tr, y_val = y_in.iloc[tr_idx], y_in.iloc[val_idx]

            sw_tr = compute_sample_weight(class_weight='balanced', y=y_tr)

            model = xgb.XGBClassifier(**xgb_params_base, random_state=seed)
            model.fit(
                X_tr, y_tr, sample_weight=sw_tr,
                eval_set=[(X_val, y_val)],
                verbose=False,
            )
            oof[val_idx] += model.predict_proba(X_val) / len(SEEDS)
            preds        += model.predict_proba(X_test_in) / (N_SPLITS * len(SEEDS))
            fold_scores.append(balanced_accuracy_score(
                y_val, np.argmax(model.predict_proba(X_val), axis=1)))
        print(f'  XGB seed {seed} | folds: {[round(s,5) for s in fold_scores]}')
    return oof, preds

print('── XGBoost Round 1 ──')
xgb_oof, xgb_preds = train_xgb(X, y, X_test)
xgb_cv = balanced_accuracy_score(y, np.argmax(xgb_oof, axis=1))
print(f'\n✅ XGBoost OOF: {xgb_cv:.5f}')

## 7. Weighted Ensemble

In [ ]:
print(f'LightGBM OOF: {lgb_cv:.5f}')
print(f'CatBoost OOF: {cb_cv:.5f}')
print(f'XGBoost  OOF: {xgb_cv:.5f}')

# ── 3-way ensemble blend search via Nelder-Mead on Balanced Accuracy ──
def neg_balacc_3(w_arr, oa, ob, oc, y_true):
    w = np.clip(w_arr, 0.0, None)
    if w.sum() < 1e-9:
        return 0.0
    w = w / w.sum()
    blended = w[0]*oa + w[1]*ob + w[2]*oc
    return -balanced_accuracy_score(y_true, np.argmax(blended, axis=1))

best = minimize(neg_balacc_3, x0=[1/3, 1/3, 1/3], args=(lgb_oof, cb_oof, xgb_oof, y),
                method='Nelder-Mead', options={'xatol': 1e-3, 'fatol': 1e-5, 'maxiter': 500})
w = np.clip(best.x, 0, None)
w = w / w.sum()
print(f'\nOptimal weights → LGB={w[0]:.3f} | CB={w[1]:.3f} | XGB={w[2]:.3f}')

ens_oof   = w[0]*lgb_oof   + w[1]*cb_oof   + w[2]*xgb_oof
ens_preds = w[0]*lgb_preds + w[1]*cb_preds + w[2]*xgb_preds

ens_score_round1 = balanced_accuracy_score(y, np.argmax(ens_oof, axis=1))
print(f'🏆 Round 1 Ensemble OOF: {ens_score_round1:.5f}')

# ============================================================
# PSEUDO-LABELING: take high-confidence test predictions and
# add them to the training data, then retrain all three models.
# ============================================================
if USE_PSEUDO_LABELING:
    test_max_proba = ens_preds.max(axis=1)
    pseudo_labels  = np.argmax(ens_preds, axis=1)
    pseudo_mask    = test_max_proba >= PSEUDO_CONF_THRESHOLD

    n_pseudo = int(pseudo_mask.sum())
    print(f'\n── Pseudo-Labeling ──')
    print(f'  Threshold: {PSEUDO_CONF_THRESHOLD}')
    print(f'  Selected {n_pseudo:,} / {len(pseudo_mask):,} test samples ({n_pseudo/len(pseudo_mask)*100:.1f}%)')
    print(f'  Pseudo-label dist: {pd.Series(pseudo_labels[pseudo_mask]).value_counts().to_dict()}')

    if n_pseudo > 1000:
        # Build augmented training set
        X_pseudo = X_test.iloc[pseudo_mask].copy()
        y_pseudo = pd.Series(pseudo_labels[pseudo_mask], index=X_pseudo.index)

        X_aug = pd.concat([X, X_pseudo], ignore_index=True)
        y_aug = pd.concat([y, y_pseudo], ignore_index=True)
        print(f'  Augmented train: {X_aug.shape}')

        # Retrain all three on augmented data
        print('\n── Round 2: retraining on augmented data ──')
        print('LightGBM Round 2:')
        lgb_oof2, lgb_preds2, _ = train_lgb(X_aug, y_aug, X_test)
        # Restrict OOF to original training rows for fair scoring
        lgb_oof2_orig = lgb_oof2[:len(y)]
        print(f'  LGB OOF (original rows): {balanced_accuracy_score(y, np.argmax(lgb_oof2_orig, axis=1)):.5f}')

        print('CatBoost Round 2:')
        cb_oof2, cb_preds2 = train_cb(X_aug, y_aug, X_test)
        cb_oof2_orig = cb_oof2[:len(y)]
        print(f'  CB OOF (original rows): {balanced_accuracy_score(y, np.argmax(cb_oof2_orig, axis=1)):.5f}')

        print('XGBoost Round 2:')
        xgb_oof2, xgb_preds2 = train_xgb(X_aug, y_aug, X_test)
        xgb_oof2_orig = xgb_oof2[:len(y)]
        print(f'  XGB OOF (original rows): {balanced_accuracy_score(y, np.argmax(xgb_oof2_orig, axis=1)):.5f}')

        # Re-optimize blend weights on the original-row OOF
        best2 = minimize(neg_balacc_3, x0=w,
                         args=(lgb_oof2_orig, cb_oof2_orig, xgb_oof2_orig, y),
                         method='Nelder-Mead', options={'xatol': 1e-3, 'fatol': 1e-5, 'maxiter': 500})
        w2 = np.clip(best2.x, 0, None)
        w2 = w2 / w2.sum()
        print(f'\nRound 2 weights → LGB={w2[0]:.3f} | CB={w2[1]:.3f} | XGB={w2[2]:.3f}')

        ens_oof_round2   = w2[0]*lgb_oof2_orig + w2[1]*cb_oof2_orig + w2[2]*xgb_oof2_orig
        ens_preds_round2 = w2[0]*lgb_preds2    + w2[1]*cb_preds2    + w2[2]*xgb_preds2

        ens_score_round2 = balanced_accuracy_score(y, np.argmax(ens_oof_round2, axis=1))
        print(f'\n🏆 Round 2 Ensemble OOF: {ens_score_round2:.5f}  (Δ {ens_score_round2 - ens_score_round1:+.5f})')

        # Take whichever round is better
        if ens_score_round2 >= ens_score_round1:
            ens_oof   = ens_oof_round2
            ens_preds = ens_preds_round2
            print('  → Using Round 2 (pseudo-labeled)')
        else:
            print('  → Round 2 worse, keeping Round 1')
    else:
        print(f'  Too few high-conf samples ({n_pseudo}), skipping pseudo-labeling')

# ── Optional per-class probability scaling ──
def neg_balacc_scale(scale, oof, y_true):
    s = np.clip(scale, 1e-3, None)
    return -balanced_accuracy_score(y_true, np.argmax(oof * s, axis=1))

res = minimize(neg_balacc_scale, x0=np.ones(3), args=(ens_oof, y),
               method='Nelder-Mead',
               options={'xatol': 1e-3, 'fatol': 1e-5, 'maxiter': 500})
class_scale = np.clip(res.x, 1e-3, None)
class_scale = class_scale / class_scale.sum() * 3
print(f'\nOptimal class scaling: {dict(zip(le_target.classes_, class_scale.round(3)))}')

ens_score_final = balanced_accuracy_score(y, np.argmax(ens_oof, axis=1))
ens_score_tuned = balanced_accuracy_score(y, np.argmax(ens_oof * class_scale, axis=1))

print('\n── Final Comparison ──')
print(f'LightGBM:           {lgb_cv:.5f}')
print(f'CatBoost:           {cb_cv:.5f}')
print(f'XGBoost:            {xgb_cv:.5f}')
print(f'Ensemble (raw):     {ens_score_final:.5f}  ← submit this if USE_THRESHOLD_TUNING=False')
print(f'Ensemble (tuned):   {ens_score_tuned:.5f}  ← submit this if USE_THRESHOLD_TUNING=True')

In [ ]:
# OOF Confidence Distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_names = le_target.classes_
colors_cls = ['#2ecc71', '#f39c12', '#e74c3c']

for i, (cls, color) in enumerate(zip(class_names, colors_cls)):
    axes[i].hist(ens_oof[:, i], bins=50, color=color, alpha=0.8, edgecolor='white')
    axes[i].set_title(f'Predicted Prob: {cls}', fontsize=11)
    axes[i].set_xlabel('Probability')
    axes[i].set_ylabel('Count')

plt.suptitle('Ensemble OOF Probability Distributions', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Submission

In [ ]:
# ── Threshold-Tuning Toggle ──
# A/B-Test: True = mit class_scale (kann auf synthetischen Daten overfitten)
#           False = raw argmax (robuster auf LB)
USE_THRESHOLD_TUNING = False

if USE_THRESHOLD_TUNING:
    final_class_ids = np.argmax(ens_preds * class_scale, axis=1)
    print(f'Using class_scale: {dict(zip(le_target.classes_, class_scale.round(3)))}')
else:
    final_class_ids = np.argmax(ens_preds, axis=1)
    print('Using raw argmax (no threshold tuning)')

final_labels = le_target.inverse_transform(final_class_ids)

sub[TARGET] = final_labels
sub.to_csv('submission.csv', index=False)

print('submission.csv saved ✅')
print(f'\nPrediction distribution:')
print(sub[TARGET].value_counts())
print(sub[TARGET].value_counts(normalize=True).round(3))
sub.head(10)

## 9. (Optional) Optuna Hyperparameter Tuning

In [ ]:
# Nur ausführen wenn du Zeit hast — dauert 30–60 min
RUN_OPTUNA = False

if RUN_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    skf_tune = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    def objective(trial):
        params = {
            'objective':         'multiclass',
            'num_class':         3,
            'metric':            'multi_logloss',
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
            'max_depth':         trial.suggest_int('max_depth', 4, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq':      1,
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
            'n_estimators':      2000,
            'random_state':      SEED,
            'verbose':           -1,
            'class_weight':      'balanced',
        }
        oof = np.zeros((len(train), 3))
        for tr_idx, val_idx in skf_tune.split(X, y):
            m = lgb.LGBMClassifier(**params)
            m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
                  eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
            oof[val_idx] = m.predict_proba(X.iloc[val_idx])
        return balanced_accuracy_score(y, np.argmax(oof, axis=1))

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    print(f'\nBest balanced accuracy: {study.best_value:.5f}')
    print(f'Best params: {study.best_params}')